# Elderly Health Monitoring AI System - Complete Pipeline
## (老人健康监测AI系统 - 完整流程)

This notebook provides a complete environment for developing, training, and evaluating the Elderly Health Monitoring AI System. It integrates data from **WESAD**, **SisFall**, and **MIMIC-IV** datasets.

### 1. Setup and Imports
First, we import the necessary libraries and define the core logic for data loading, preprocessing, and model architectures.

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import joblib
from tqdm.auto import tqdm

print("TensorFlow version:", tf.__version__)
print("Eager execution:", tf.executing_eagerly())

### 2. Core Classes (Data Ingestion & Preprocessing)
We define the `DataLoader` and `DataPreprocessor` classes directly in the notebook for easy modification.

In [ ]:
class DataLoader:
    def __init__(self, base_path='../data/'):
        self.base_path = base_path

    def load_wesad_subject(self, subject_id):
        subject_path = os.path.join(self.base_path, 'WESAD', subject_id, f'{subject_id}.pkl')
        if not os.path.exists(subject_path):
            return None
        with open(subject_path, 'rb') as f:
            data = pickle.load(f, encoding='latin1')
        return data['signal']['chest'], data['label']

    def load_sisfall_file(self, subject_id, filename):
        file_path = os.path.join(self.base_path, 'SisFall', subject_id, filename)
        if not os.path.exists(file_path): return None
        df = pd.read_csv(file_path, header=None, sep=',', skipinitialspace=True)
        df.iloc[:, -1] = df.iloc[:, -1].astype(str).str.replace(';', '').astype(float)
        label = 1 if filename.startswith('F') else 0
        return df, label

class DataPreprocessor:
    def normalize_series(self, data):
        mean = np.mean(data, axis=0)
        std = np.std(data, axis=0)
        return (data - mean) / (std + 1e-7)

    def create_sequences(self, data, labels=None, seq_length=60, step=30):
        sequences, target_labels = [], []
        for i in range(0, len(data) - seq_length, step):
            sequences.append(data[i:i + seq_length])
            if labels is not None:
                window_labels = labels[i:i + seq_length]
                target_labels.append(np.argmax(np.bincount(window_labels.flatten().astype(int))))
        return (np.array(sequences), np.array(target_labels)) if labels is not None else np.array(sequences)

### 3. Model Architectures
Defining the LSTM for health risk and Random Forest for fall detection.

In [ ]:
class HealthRiskModel:
    def __init__(self, input_shape):
        self.model = models.Sequential([
            layers.LSTM(64, input_shape=input_shape, return_sequences=True),
            layers.Dropout(0.2),
            layers.LSTM(32),
            layers.Dense(16, activation='relu'),
            layers.Dense(3, activation='softmax')
        ])
        self.model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

class FallDetectionModel:
    def __init__(self):
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)

### 4. The Complete Execution Loop
This is where we process all subjects and train the final models.

In [ ]:
loader = DataLoader(base_path='../data/')
preprocessor = DataPreprocessor()

# --- WESAD Processing ---
wesad_subjects = ['S2', 'S3'] # Using first 2 subjects for speed in this demo
X_all, y_all = [], []
for sub in tqdm(wesad_subjects, desc="WESAD Ingestion"):
    data = loader.load_wesad_subject(sub)
    if data:
        signals, labels = data
        norm_acc = preprocessor.normalize_series(signals['ACC'])
        X, y = preprocessor.create_sequences(norm_acc, labels, seq_length=700, step=700)
        X_all.append(X); y_all.append(y)

X_wesad = np.vstack(X_all)
y_wesad = np.concatenate(y_all)
X_train, X_test, y_train, y_test = train_test_split(X_wesad, y_wesad, test_size=0.2)

# --- LSTM Training ---
print("\nTraining Health Risk LSTM...")
risk_model = HealthRiskModel(input_shape=(700, 3))
risk_model.model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

### 5. Final Saving
Persisting the models for use in the Flask API.

In [ ]:
os.makedirs('../models', exist_ok=True)
risk_model.model.save('../models/health_risk_final.h5')
print("Model saved successfully!")